In [3]:
import sys
sys.path.insert(0, 'C:/Users/pale4/projects/factor-research-platform')

from config.settings import engine
from factors.momentum import Momentum

mom = Momentum("momentum", engine)
raw = mom.compute()
print(raw.head(20))
print(raw.dropna().shape)

                         date ticker  raw_score
0   2021-02-01 00:00:00-06:00      A        NaN
1   2021-03-01 00:00:00-06:00      A        NaN
2   2021-04-01 00:00:00-06:00      A        NaN
3   2021-05-01 00:00:00-05:00      A        NaN
4   2021-06-01 00:00:00-05:00      A        NaN
5   2021-07-01 00:00:00-05:00      A        NaN
6   2021-08-01 00:00:00-05:00      A        NaN
7   2021-09-01 00:00:00-05:00      A        NaN
8   2021-10-01 00:00:00-05:00      A        NaN
9   2021-11-01 00:00:00-06:00      A        NaN
10  2021-12-01 00:00:00-06:00      A        NaN
11  2022-01-01 00:00:00-06:00      A   0.337190
12  2022-02-01 00:00:00-06:00      A   0.190550
13  2022-03-01 00:00:00-06:00      A   0.077452
14  2022-04-01 00:00:00-06:00      A   0.031033
15  2022-05-01 00:00:00-05:00      A  -0.113536
16  2022-06-01 00:00:00-05:00      A  -0.122185
17  2022-07-01 00:00:00-05:00      A  -0.184135
18  2022-08-01 00:00:00-05:00      A  -0.199390
19  2022-09-01 00:00:00-05:00      A  -0

In [4]:
normalized = mom.normalize(raw.dropna())
print(normalized.head())

with_quintiles = mom.assign_quintiles(normalized)
print(with_quintiles.head())

                         date ticker  raw_score   z_score
11  2022-01-01 00:00:00-06:00      A   0.337190  0.586653
12  2022-02-01 00:00:00-06:00      A   0.190550  0.266591
13  2022-03-01 00:00:00-06:00      A   0.077452  0.011836
14  2022-04-01 00:00:00-06:00      A   0.031033 -0.240554
15  2022-05-01 00:00:00-05:00      A  -0.113536 -0.504126
                         date ticker  raw_score   z_score quintile
11  2022-01-01 00:00:00-06:00      A   0.337190  0.586653        4
12  2022-02-01 00:00:00-06:00      A   0.190550  0.266591        3
13  2022-03-01 00:00:00-06:00      A   0.077452  0.011836        3
14  2022-04-01 00:00:00-06:00      A   0.031033 -0.240554        3
15  2022-05-01 00:00:00-05:00      A  -0.113536 -0.504126        2


In [6]:
import sys
sys.path.insert(0, 'C:/Users/pale4/projects/factor-research-platform')

import pandas as pd
from config.settings import engine
from factors.base import Factor


class Size(Factor):
    def compute(self):
        derived_df = pd.read_sql("SELECT * FROM derived_fundamentals", self.engine)
        derived_df = derived_df.rename(columns={"report_date": "date"})
        derived_df["date"] = pd.to_datetime(derived_df["date"])
        monthly_df = pd.read_sql("SELECT * FROM monthly_returns", self.engine)
        monthly_df = monthly_df.rename(columns={"month_start": "date"})
        monthly_df["date"] = pd.to_datetime(monthly_df["date"])
        data = pd.merge_asof(
            monthly_df.sort_values("date"),
            derived_df.sort_values("date"),
            on="date",
            by="ticker",
            direction="backward",
        )
        data["raw_score"] = -data["market_cap"]
        return data[["date", "ticker", "raw_score"]]



In [8]:
derived_df = pd.read_sql("SELECT * FROM derived_fundamentals", engine)
monthly_df = pd.read_sql("SELECT * FROM monthly_returns", engine)

In [11]:
print(monthly_df['month_start'].dtype)
print(derived_df['report_date'].dtype)

object
object


In [4]:
import sys
sys.path.insert(0, 'C:/Users/pale4/projects/factor-research-platform')

from config.settings import engine
from factors.size import Size
from factors.value import Value

size = Size("size", engine)
val = Value("value", engine)

size_raw = size.compute()
value_raw = val.compute()

print(size_raw.dropna().shape)
print(value_raw.dropna().shape)
print(value_raw.dropna().head(20))

(7661, 3)
(7661, 3)
            date ticker    raw_score
20653 2024-07-01   EXPD     0.126284
20860 2024-07-01  BRK-B  1029.513437
20973 2024-08-01   EXPD     0.126284
21181 2024-08-01  BRK-B  1029.513437
21372 2024-09-01  BRK-B  1029.513437
21840 2024-09-01   EXPD     0.126284
21869 2024-10-01   VTRS     1.514565
21871 2024-10-01    CNP     0.569583
21872 2024-10-01  BRK-B   950.724247
21878 2024-10-01   TRMB     0.384957
21880 2024-10-01   DASH     0.127734
21883 2024-10-01   SCHW     0.405818
21884 2024-10-01   FSLR     0.284359
21888 2024-10-01    WRB     0.403925
21890 2024-10-01    APP     0.021422
21892 2024-10-01     IP     0.533546
21894 2024-10-01    AVY     0.138281
21897 2024-10-01   IBKR     0.275007
21900 2024-10-01    WTW     0.255736
21903 2024-10-01   DDOG     0.067351


In [10]:
print(derived_df[derived_df['ticker'] == 'BRK-B'][['ticker', 'report_date', 'stockholders_equity', 'market_cap']])

    ticker report_date  stockholders_equity    market_cap
335  BRK-B  2024-06-30         6.016970e+11  5.844479e+08
336  BRK-B  2024-09-30         6.290690e+11  6.616735e+08
337  BRK-B  2024-12-31         6.493680e+11  6.519177e+08
338  BRK-B  2025-03-31         6.544710e+11  7.659688e+08
339  BRK-B  2025-06-30         6.679890e+11  6.986456e+08


In [11]:
print(derived_df[derived_df['ticker'] == 'BRK-B'][['ticker', 'report_date', 'ordinary_shares_number']])

    ticker report_date  ordinary_shares_number
335  BRK-B  2024-06-30               1436696.0
336  BRK-B  2024-09-30               1437608.0
337  BRK-B  2024-12-31               1438223.0
338  BRK-B  2025-03-31               1438223.0
339  BRK-B  2025-06-30               1438223.0


In [12]:
print(derived_df[derived_df['ticker'] == 'AAPL'][['ticker', 'report_date', 'ordinary_shares_number', 'market_cap']])

  ticker report_date  ordinary_shares_number    market_cap
5   AAPL  2024-12-31            1.503787e+10  3.745325e+12
6   AAPL  2025-03-31            1.493932e+10  3.304070e+12
7   AAPL  2025-06-30            1.485672e+10  3.038905e+12
8   AAPL  2025-09-30            1.477326e+10  3.754559e+12
9   AAPL  2025-12-31            1.469793e+10  3.992042e+12


In [5]:
value_norm = val.normalize(value_raw.dropna())
value_q = val.assign_quintiles(value_norm)
val.store(value_q)

In [14]:
import sys
sys.path.insert(0, 'C:/Users/pale4/projects/factor-research-platform')

from config.settings import engine
from factors.volatility import Volatility
from factors.quality import Quality 

vol = Volatility('volatility', engine)
qua = Quality('quality', engine)

raw_vol = vol.compute() 
raw_qua = qua.compute()

print(raw_vol.dropna().shape)
print(raw_qua.dropna().shape)

(24375, 3)
(7668, 3)


In [15]:
vol_norm = vol.normalize(raw_vol.dropna())
qua_norm = qua.normalize(raw_qua.dropna())

vol_q = vol.assign_quintiles(vol_norm)
qua_q = qua.assign_quintiles(qua_norm)

vol.store(vol_q)
qua.store(qua_q)